# 01 — Data Inventory and Validation
## Trade Winds & Headwinds: Twenty-Five Years of Tourism and Economic Change in the USVI

**Purpose:** Establish a trustworthy analytical foundation before any substantive analysis.  
This notebook:
1. Profiles every raw data file (coverage, shape, units, missingness)
2. Runs cross-source validation checks
3. Produces a prioritised data-quality log
4. Documents the Census NAICS crosswalk
5. Generates clean interim datasets

**Rule:** Do not overwrite or modify raw files.  All outputs go to `data/interim/`.

In [1]:
import sys
import os
from pathlib import Path

# Ensure src/ is importable regardless of working directory
PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import warnings
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.1f}".format)

print(f"Project root: {PROJ_ROOT}")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

Project root: C:\Users\johbr\Data-Analyst\Projects\Trade-Winds-and-Headwinds
pandas 3.1.0  |  numpy 2.5.4


---
## Section 1 — NPS Recreation Visitation

In [2]:
from src.cleaning import load_nps_annual, load_nps_monthly

# --- Annual ---
viis_ann = load_nps_annual("VIIS")
buis_ann = load_nps_annual("BUIS")

print("=== VIIS Annual ===")
print(
    f"  Rows: {len(viis_ann)}  |  Years: {viis_ann['year'].min()}–{viis_ann['year'].max()}"
)
buis_years = buis_ann["year"].sort_values().tolist()
gaps = [
    (buis_years[i - 1] + 1, buis_years[i] - 1)
    for i in range(1, len(buis_years))
    if buis_years[i] - buis_years[i - 1] > 1
]
print("\n=== BUIS Annual ===")
print(
    f"  Rows: {len(buis_ann)}  |  Years: {buis_ann['year'].min()}–{buis_ann['year'].max()}"
)
print(f"  Gaps: {gaps}")

print("\n=== Recent annual values ===")
recent = (
    pd.concat([viis_ann, buis_ann])
    .query("year >= 2015")
    .pivot_table(index="year", columns="park", values="recreation_visitors")
    .sort_index()
)
print(recent.to_string())

=== VIIS Annual ===
  Rows: 69  |  Years: 1957–2025

=== BUIS Annual ===
  Rows: 61  |  Years: 1962–2025
  Gaps: [(1999, 2001)]

=== Recent annual values ===
park     BUIS      VIIS
year                   
2015 41,936.0 438,372.0
2016 38,929.0 411,343.0
2017 33,082.0 304,408.0
2018 31,411.0 112,287.0
2019 37,085.0 133,398.0
2020 16,032.0 167,540.0
2021 45,664.0 323,999.0
2022 45,365.0 196,752.0
2023 46,207.0 343,685.0
2024 42,289.0 423,029.0
2025 40,079.0 471,074.0


In [3]:
# --- Monthly ---
viis_mo = load_nps_monthly("VIIS")
buis_mo = load_nps_monthly("BUIS")

print("=== VIIS Monthly ===")
print(
    f"  Rows: {len(viis_mo)}  |  Range: {viis_mo['date'].min()} – {viis_mo['date'].max()}"
)
print(f"  Non-null months: {viis_mo['recreation_visitors'].notna().sum()}")
print(f"  Null months: {viis_mo['recreation_visitors'].isna().sum()}")

# Flag 2026 anomaly
viis_2026 = viis_mo[viis_mo["year"] == 2026].dropna(subset=["recreation_visitors"])
if not viis_2026.empty:
    print(f"\n  ⚠ 2026 partial data (check for methodology change):")
    print(viis_2026[["date", "recreation_visitors"]].to_string(index=False))

# Annual-vs-monthly validation
from src.validation import validate_nps_annual_vs_monthly

disc_viis = validate_nps_annual_vs_monthly(viis_ann, viis_mo, "VIIS")
disc_buis = validate_nps_annual_vs_monthly(buis_ann, buis_mo, "BUIS")
print(f"\nVIIS annual vs. monthly sum discrepancies: {len(disc_viis)} year(s)")
if not disc_viis.empty:
    print(
        disc_viis[["year", "annual_value", "monthly_sum", "discrepancy_pct"]].to_string(
            index=False
        )
    )
print(f"\nBUIS annual vs. monthly sum discrepancies: {len(disc_buis)} year(s)")
if not disc_buis.empty:
    print(
        disc_buis[["year", "annual_value", "monthly_sum", "discrepancy_pct"]]
        .head(10)
        .to_string(index=False)
    )

=== VIIS Monthly ===
  Rows: 564  |  Range: 1979-01 – 2025-12
  Non-null months: 564
  Null months: 0

VIIS annual vs. monthly sum discrepancies: 0 year(s)

BUIS annual vs. monthly sum discrepancies: 0 year(s)


---
## Section 2 — FRED Employment

In [4]:
from src.cleaning import load_fred_employment

fred = load_fred_employment()
print("=== FRED Leisure & Hospitality Employment ===")
print(f"  Rows: {len(fred)}  |  Range: {fred['date'].min()} – {fred['date'].max()}")
print(f"  Units: thousands of persons")
print(f"  SA  — min: {fred['emp_sa'].min():.1f}k  max: {fred['emp_sa'].max():.1f}k")
print(f"  NSA — min: {fred['emp_nsa'].min():.1f}k  max: {fred['emp_nsa'].max():.1f}k")
print(
    f"  Null SA: {fred['emp_sa'].isna().sum()}  |  Null NSA: {fred['emp_nsa'].isna().sum()}"
)

print("\n=== Key period annual averages (thousands) ===")
annual_avg = (
    fred.groupby("year")[["emp_sa", "emp_nsa"]]
    .mean()
    .round(2)
    .rename(columns={"emp_sa": "SA_avg", "emp_nsa": "NSA_avg"})
)
print(annual_avg.query("year >= 2015").to_string())

=== FRED Leisure & Hospitality Employment ===
  Rows: 441  |  Range: 1990-01 – 2026-09
  Units: thousands of persons
  SA  — min: 3.7k  max: 7.8k
  NSA — min: 3.9k  max: 8.0k
  Null SA: 0  |  Null NSA: 0

=== Key period annual averages (thousands) ===
      SA_avg  NSA_avg
year                 
2015     7.4      7.4
2016     7.7      7.7
2017     6.9      6.9
2018     4.2      4.2
2019     5.6      5.6
2020     4.9      4.9
2021     5.8      5.8
2022     6.3      6.3
2023     6.8      6.8
2024     7.0      7.0
2025     6.5      6.5
2026     5.5      5.5


---
## Section 3 — USVIBER Tourism Indicators (TOUR24)

In [5]:
from src.cleaning import load_usviber_tour24
from src.validation import validate_usviber_arithmetic

tour = load_usviber_tour24()

print("=== USVIBER TOUR24 — shape and coverage ===")
print(f"  Years: {tour['year'].min()}–{tour['year'].max()}  ({len(tour)} rows)")
print(f"\n  Key columns (first/last 3 years):")
cols_show = [
    "year",
    "total_visitors",
    "tourists",
    "excursionists",
    "air_excursionists",
    "cruise_passengers",
    "air_visitors",
    "expenditures_total",
    "total_rooms",
    "occupancy_rate_pct",
]
print(tour[cols_show].set_index("year").to_string())

=== USVIBER TOUR24 — shape and coverage ===
  Years: 2000–2024  (16 rows)

  Key columns (first/last 3 years):
      total_visitors  tourists  excursionists  air_excursionists  cruise_passengers  air_visitors  expenditures_total  total_rooms  occupancy_rate_pct
year                                                                                                                                                  
2000         2,516.9     546.7        1,970.2              216.7            1,768.4         604.5             1,186.1         5008                57.7
2010         2,697.7     576.2        2,121.5              112.9            1,858.9         679.4             1,568.7         5008                57.1
2011         2,688.4     566.9        2,121.5              112.5            2,009.0         679.4             1,642.4         4889                51.6
2012         2,646.3     619.2        2,027.2              122.7            1,904.5         741.9             1,732.6         5030    

In [6]:
print("=== USVIBER arithmetic consistency check ===")
print("(Checks: totals = parts; tolerance ±0.5k)\n")
flagged = validate_usviber_arithmetic(tour)
if flagged.empty:
    print("  ✓ All checks within rounding tolerance.")
else:
    print(flagged.to_string(index=False))

print("\n=== Data quality flags ===")
dq_cols = [c for c in tour.columns if "_dq_flag" in c]
for col in dq_cols:
    flagged_rows = tour[tour[col].notna()]
    if not flagged_rows.empty:
        print(f"  {col}:")
        for _, row in flagged_rows.iterrows():
            print(f"    Year {row['year']}: {row[col]}")

=== USVIBER arithmetic consistency check ===
(Checks: totals = parts; tolerance ±0.5k)

 year                        check  actual  expected  diff      status
 2000 excursionists = air + cruise 1,970.2   1,985.1 -14.9 discrepancy
 2010 excursionists = air + cruise 2,121.5   1,971.8 149.7 discrepancy

=== Data quality flags ===
  air_excursionists_dq_flag:
    Year 2020: 2020 value (216.7k) = 61.6% of air arrivals as same-day; implausible for COVID year; arithmetic reconciles but value is suspect


---
## Section 4 — World Bank Tourism Data

In [7]:
from src.cleaning import load_worldbank

wb_arr = load_worldbank("arrivals")
wb_rcpt = load_worldbank("receipts")

print("=== World Bank Arrivals ===")
print(f"  Non-null years: {wb_arr['value'].notna().sum()}")
print(
    f"  Coverage (non-null): {wb_arr.dropna()['year'].min()}–{wb_arr.dropna()['year'].max()}"
)
print(
    f"  DQ flagged rows: {wb_arr['dq_flag'].notna().sum() if 'dq_flag' in wb_arr else 0}"
)
print(wb_arr.dropna(subset=["value"]).tail(10).to_string(index=False))

print("\n=== World Bank Receipts (million USD) ===")
print(f"  Non-null years: {wb_rcpt['value'].notna().sum()}")
print(
    wb_rcpt.dropna(subset=["value"])
    .assign(value_mUSD=lambda d: d["value"] / 1e6)
    .tail(10)[["year", "value_mUSD"]]
    .to_string(index=False)
)

# Cross-check with USVIBER expenditures (both are in roughly similar units)
print("\n=== Cross-check: World Bank receipts vs USVIBER total expenditures ===")
comp = tour[["year", "expenditures_total"]].merge(
    wb_rcpt[["year", "value"]].rename(columns={"value": "wb_receipts_USD"}),
    on="year",
    how="inner",
)
comp["wb_mUSD"] = comp["wb_receipts_USD"] / 1e6
comp["ratio_wb_usviber"] = (comp["wb_mUSD"] / comp["expenditures_total"]).round(2)
print(
    comp[["year", "expenditures_total", "wb_mUSD", "ratio_wb_usviber"]].to_string(
        index=False
    )
)
print(
    "\nNote: Ratio ~1.0 = consistent; >1 = WB higher (may include non-tourist receipts)."
)

=== World Bank Arrivals ===
  Non-null years: 25
  Coverage (non-null): nan–nan
  DQ flagged rows: 1
 year       value indicator dq_flag
 2010 2,531,000.0  arrivals     NaN
 2011 2,686,000.0  arrivals     NaN
 2012 2,640,000.0  arrivals     NaN
 2013 2,702,000.0  arrivals     NaN
 2014 2,814,000.0  arrivals     NaN
 2015 2,643,000.0  arrivals     NaN
 2016 2,574,000.0  arrivals     NaN
 2017 1,954,000.0  arrivals     NaN
 2018 1,922,000.0  arrivals     NaN
 2019 2,074,000.0  arrivals     NaN

=== World Bank Receipts (million USD) ===
  Non-null years: 26
 year  value_mUSD
 2011     1,122.0
 2012     1,210.0
 2013     1,168.0
 2014     1,319.0
 2015     1,324.0
 2016     1,343.0
 2017     1,202.0
 2018       993.0
 2019     1,031.0
 2020       686.0

=== Cross-check: World Bank receipts vs USVIBER total expenditures ===
 year  expenditures_total  wb_mUSD  ratio_wb_usviber
 2000             1,186.1  1,206.0               1.0
 2010             1,568.7  1,223.0               0.8
 2011     

C:\Users\johbr\Data-Analyst\Projects\Trade-Winds-and-Headwinds\src\cleaning.py:237: UserWarning: World Bank arrivals: 1 anomalous value (>5M) set to NaN. Verify source before use.
  warnings.warn(


---
## Section 5 — U.S. Census Economic Census

In [8]:
import importlib
import src.cleaning as _cleaning_mod

importlib.reload(_cleaning_mod)

from src.cleaning import (
    load_census_2007,
    load_census_2012_basic,
    load_census_2017_basic,
    load_census_2022_basic,
    load_census_rooms,
)

c07 = load_census_2007()
c12 = load_census_2012_basic()
c17 = load_census_2017_basic()
c22 = load_census_2022_basic()


# Filter to NAICS 72 across all vintages
def naics72(df):
    return df[df["naics"].astype(str).str.startswith("72")].copy()


print("=== Census NAICS 72 — Accommodation & Food Services ===\n")
frames = []
for df, label in [
    (c07, "2007-file"),
    (c12, "2012"),
    (c17, "2017-file"),
    (c22, "2022-file"),
]:
    sub = naics72(df)[
        ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
    ].copy()
    sub["vintage"] = label
    frames.append(sub)

combined = pd.concat(frames, ignore_index=True)
top_rows = combined[combined["naics"].isin(["72", "721", "722"])].sort_values(
    ["naics", "year"]
)
print(top_rows.to_string(index=False))

=== Census NAICS 72 — Accommodation & Food Services ===

 year naics                       naics_label  estab  rcptot_k  payann_k     emp   vintage
 2002    72   Accommodation and food services    313 331,008.0  92,357.0 5,639.0 2007-file
 2007    72   Accommodation and food services    255 460,532.0 129,635.0 6,146.0 2007-file
 2012    72   Accommodation and food services    279 539,442.0 160,542.0 6,864.0      2012
 2012    72   Accommodation and food services    279 539,442.0 160,542.0 6,864.0 2017-file
 2017    72   Accommodation and food services    304 520,417.0 149,735.0 6,737.0 2017-file
 2017    72   Accommodation and food services    304 520,417.0 149,735.0 6,737.0 2022-file
 2022    72   Accommodation and food services    293 536,134.0 136,885.0 5,140.0 2022-file
 2012   721                     Accommodation     57 325,265.0 111,879.0 3,876.0      2012
 2017   721                     Accommodation     51 320,666.0  99,222.0 3,385.0 2022-file
 2022   721                     A

In [9]:
importlib.reload(_cleaning_mod)
from src.cleaning import load_census_rooms

print("=== Guestroom count series across Census vintages ===\n")
rooms = load_census_rooms()
print(
    rooms[["year", "rooms", "estab", "rcptot_k", "methodology_note"]].to_string(
        index=False
    )
)

print("\n⚠ Methodology note:")
print("  2007/2012 use the A10 industry-total row.")
print("  2017/2022 use NAPCS product 7003825000 (Room accommodation).")
print("  These are the most comparable measures available; cross-vintage")
print("  comparability is approximate, not exact.")
print("\n  IMPORTANT: 2017 reference date = Dec 31, 2017 (post-hurricane).")
print("  The 2017 room count reflects storm-damaged/closed inventory,")
print("  not the pre-hurricane capacity.")

=== Guestroom count series across Census vintages ===

 year  rooms  estab  rcptot_k                                                                                                                methodology_note
 2007   4065     47    308481                                                                                               A10 industry total (PSCODE 20000)
 2012   5027     54    319049                                                                                         A10 industry total (PSCODE2012 = 20000)
 2017   2624     46    237838 NAPCS02 product 7003825000 (Room or unit accommodation for travelers). Guestroom reference date = Dec 31, 2017, post-hurricane.
 2022   2039     51    135204                                                           NAPCS02 product 7003825000 (Room or unit accommodation for travelers)

⚠ Methodology note:
  2007/2012 use the A10 industry-total row.
  2017/2022 use NAPCS product 7003825000 (Room accommodation).
  These are the most compar

### Census NAICS crosswalk

The table below classifies Census comparisons across the 2007–2022 vintages.

| NAICS level | 2007 code | 2012 code | 2017 code | 2022 code | Comparability |
|---|---|---|---|---|---|
| Accommodation & food services (2-digit) | 72 | 72 | 72 | 72 | **Directly comparable** |
| Accommodation (3-digit) | — (in A10 only as 7211) | 721 | 721 | 721 | **Comparable after aggregation** |
| Food services (3-digit) | — | 722 | 722 | 722 | **Directly comparable** |
| Hotels & motels (4-digit) | 7211 | 7211 | 7211 | 7211 | **Directly comparable** (small number of establishments — watch for suppression) |
| Guestrooms (RMRTRAN) | A10 total row | A10 total row | NAPCS 7003825000 | NAPCS 7003825000 | **Comparable after documentation** — different table structures but conceptually equivalent |
| Restaurants (5-digit) | — | 7225 | 7225 | 7225 | **Directly comparable** |

**Not comparable without further work:** Any NAICS code that changed definition between NAICS 2007, 2012, 2017, or 2022.  For this project the 2-digit aggregates are the most defensible cross-vintage comparisons.

---
## Section 6 — BEA GDP and Industry Value Added

In [10]:
importlib.reload(_cleaning_mod)
from src.cleaning import load_bea

# Table 2.1: nominal value added by industry
bea_va = load_bea("2.1")
bea_pct = load_bea("2.2")

print("=== BEA Table 2.1 — Value Added by Industry (nominal, $M) ===")
print(f"  Rows: {len(bea_va)}  |  Years: {sorted(bea_va['year'].unique())}")
print()

# Accommodation and food services
af = bea_va[bea_va["label"].str.contains("ccommodation", case=False, na=False)]
print("Accommodation and food services value added ($M):")
print(af[["year", "value", "vintage_flag"]].sort_values("year").to_string(index=False))

# GDP total
gdp = bea_va[
    bea_va["label"].str.contains("ross domestic product", case=False, na=False)
]
print("\nGDP ($M):")
print(gdp[["year", "value", "vintage_flag"]].sort_values("year").to_string(index=False))

=== BEA Table 2.1 — Value Added by Industry (nominal, $M) ===
  Rows: 200  |  Years: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]

Accommodation and food services value added ($M):
 year  value vintage_flag
 2002  267.0             
 2003  284.0             
 2004  308.0             
 2005  325.0             
 2006  330.0             
 2007  405.0             
 2008  424.0             
 2009  400.0             
 2010  428.0             
 2011  436.0             
 2012  450.0             
 2013  454.0             
 2014  470.0             
 2015  486.0             
 2016  483.0             
 2017  436.0             
 2018  288.0            r
 2019  341.0            r
 2020  253.0            r
 2021  495.0           

In [11]:
# Real value added (Table 2.3)
importlib.reload(_cleaning_mod)
from src.cleaning import load_bea

bea_real = load_bea("2.3")
af_real = bea_real[bea_real["label"].str.contains("ccommodation", case=False, na=False)]
print("=== Real Accommodation & Food Services Value Added (chained $M) ===")
print(
    af_real[["year", "value", "vintage_flag"]]
    .sort_values("year")
    .to_string(index=False)
)

# GDP share: compute from Table 2.1 data
gdp_nom = bea_va[
    bea_va["label"].str.contains("ross domestic product", case=False, na=False)
    & ~bea_va["label"].str.contains("Private", case=False, na=False)
    & ~bea_va["label"].str.contains("Government", case=False, na=False)
]
# Keep only the top-level GDP row (line "1")
gdp_top = gdp_nom[gdp_nom["line"] == "1"][["year", "value"]].rename(
    columns={"value": "gdp_nom"}
)
af_nom = bea_va[bea_va["label"].str.contains("ccommodation", case=False, na=False)][
    ["year", "value"]
].rename(columns={"value": "af_nom"})
gdp_share = af_nom.merge(gdp_top, on="year").copy()
gdp_share["af_pct_gdp"] = (gdp_share["af_nom"] / gdp_share["gdp_nom"] * 100).round(1)
print("\n=== GDP share of Accommodation & Food Services (%, computed) ===")
print(
    gdp_share[["year", "af_nom", "gdp_nom", "af_pct_gdp"]]
    .sort_values("year")
    .to_string(index=False)
)

=== Real Accommodation & Food Services Value Added (chained $M) ===
 year  value vintage_flag
 2002  342.0             
 2003  360.0             
 2004  380.0             
 2005  387.0             
 2006  382.0             
 2007  450.0             
 2008  470.0             
 2009  416.0             
 2010  445.0             
 2011  454.0             
 2012  450.0             
 2013  441.0             
 2014  444.0             
 2015  438.0             
 2016  428.0             
 2017  376.0             
 2018  240.0            r
 2019  274.0            r
 2020  200.0            r
 2021  373.0             

=== GDP share of Accommodation & Food Services (%, computed) ===
 year  af_nom  gdp_nom  af_pct_gdp
 2002   267.0  3,262.0         8.2
 2003   284.0  3,443.0         8.2
 2004   308.0  3,797.0         8.1
 2005   325.0  4,428.0         7.3
 2006   330.0  4,484.0         7.4
 2007   405.0  4,784.0         8.5
 2008   424.0  4,244.0        10.0
 2009   400.0  4,201.0         9.5
 2010

---
## Section 7 — USVI CPI

In [12]:
from src.cleaning import load_usvi_cpi

cpi = load_usvi_cpi()
print("=== USVI CPI (2009 = 100, from USVIBER Econ21.pdf) ===")
print(cpi[["year", "cpi_usvi_2009_100"]].to_string(index=False))

print("\n⚠ Coverage: 2009–2021 only.  No reliable USVI CPI available post-2021.")
print("  The 2018 drop from 130.0 → 99.8 is anomalous and should be verified")
print("  against the original Econ21.pdf table layout before use.")
print("\n  Impact: Cannot deflate nominal monetary series (visitor expenditures,")
print("  BEA value added) beyond 2021 without a defensible deflator.")
print("  All nominal series preserved. Real-dollar analysis is limited to")
print("  BEA's own chained-dollar measures (Table 2.3).")

=== USVI CPI (2009 = 100, from USVIBER Econ21.pdf) ===
 year  cpi_usvi_2009_100
 2009              100.0
 2010              104.0
 2011              109.7
 2012              115.7
 2013              118.5
 2014              119.3
 2015              121.7
 2016              123.8
 2017              130.0
 2018               99.8
 2019              102.8
 2020              109.6
 2021              115.8

⚠ Coverage: 2009–2021 only.  No reliable USVI CPI available post-2021.
  The 2018 drop from 130.0 → 99.8 is anomalous and should be verified
  against the original Econ21.pdf table layout before use.

  Impact: Cannot deflate nominal monetary series (visitor expenditures,
  BEA value added) beyond 2021 without a defensible deflator.
  All nominal series preserved. Real-dollar analysis is limited to
  BEA's own chained-dollar measures (Table 2.3).


---
## Section 8 — Data Quality Log

In [13]:
from src.validation import DATA_QUALITY_LOG, print_dq_log

dq_df = pd.DataFrame(
    [
        {
            "severity": issue.severity,
            "source": issue.source,
            "variable": issue.variable,
            "evidence": (
                issue.evidence[:90] + "..."
                if len(issue.evidence) > 90
                else issue.evidence
            ),
            "treatment": (
                issue.recommended_treatment[:90] + "..."
                if len(issue.recommended_treatment) > 90
                else issue.recommended_treatment
            ),
        }
        for issue in DATA_QUALITY_LOG
    ]
).sort_values(["severity", "source"])


def sev_style(val):
    colors = {
        "HIGH": "background-color: #ffe0e0",
        "MEDIUM": "background-color: #fff3cd",
        "LOW": "background-color: #e8f4f8",
    }
    return colors.get(val, "")


print(
    f"Total issues: {len(dq_df)}  |  HIGH: {(dq_df.severity=='HIGH').sum()}  |  MEDIUM: {(dq_df.severity=='MEDIUM').sum()}  |  LOW: {(dq_df.severity=='LOW').sum()}"
)
# applymap was renamed to map in pandas 2.1+
dq_df.style.map(sev_style, subset=["severity"])

Total issues: 14  |  HIGH: 2  |  MEDIUM: 8  |  LOW: 4


,severity,source,variable,evidence,treatment
1,HIGH,VIIS Recreation Visitors By Month (2026),monthly recreation visitors (Jan–Aug 2026),"Jan–Jun 2026 each show 100,000–170,000 visitors, roughly 3× the corresponding 2025 months ...",Exclude 2026 from any trend analysis until the source NPS portal is re-checked for a metho...
0,HIGH,World Bank / VIR.ST.INT.ARVL.json,arrivals (2020),"2020 value = 8,612,000, which is ~4× the normal range (~2M) and physically impossible in a...",Set to NaN. Do not use for analysis until verified against World Bank source API or USVIBE...
13,LOW,BEA vigdp0624.xlsx,industry value-added coverage,Table 1.x (GDP expenditure accounts): 2002–2022. Tables 2.x (industry value added): 2002–2...,Use Table 1.1 for nominal GDP through 2022. Use Tables 2.x for accommodation value added t...
10,LOW,FRED SMS78000007000000001 / SMU78000007000000001,"emp_sa, emp_nsa — units","Values are in thousands of persons (e.g., 8.0 = 8,000 workers). This must be labeled clear...","Always label axes as 'thousands of employees' or multiply by 1,000 before comparing with C..."
12,LOW,USVI CPI / Econ21.pdf,cpi_2018 value,"The CPI index drops from 130.0 (2017) to 99.8 (2018), a ~23% fall, which seems implausible...",Flag in data dictionary. Do not use CPI-deflated series spanning 2017–2018 without verifyi...
11,LOW,World Bank JSON (both files),pagination,Metadata shows pages=2 (total=66 records) but local file contains only page 1 (50 records)...,Pre-1995 data is not needed for the 2000–2025 analysis scope. No action required.
7,MEDIUM,BUIS Annual / Monthly,recreation_visitors (1999–2001),"Years 1999, 2000, and 2001 are absent from both BUIS annual and monthly files. The monthly...",Note gap in visualisations. Do not interpolate. The NPS IRMA portal may have data; conside...
8,MEDIUM,Census 2017 NAPCS02,guestrooms (reference date),"Census guestroom count is as of December 31 of the census year. For 2017, this is ~3 month...",Label as 'end-of-2017 (post-hurricane)' in all tables and charts. Do not use as a pre-hurr...
6,MEDIUM,"Census NAPCS02 (2017, 2022)",RMRTRAN (guestrooms) in Total row vs room-product row,"2017 NAPCS02 Total row: RMRTRAN = 8,735. 2017 NAPCS02 product 7003825000 row: RMRTRAN = 2,...","Use product 7003825000 RMRTRAN for 2017 and 2022 guestroom counts. For 2007 and 2012, use ..."
4,MEDIUM,NPS VIIS Recreation Visitors By Month,counting methodology (three periods),"2003–2021 method: campground + Trunk Bay multipliers (8.53 persons/boat), beach and visito...",Exclude 2026 from historical comparisons. Document the 2022 method change as a secondary m...


---
## Section 9 — Save Interim Datasets

In [14]:
INTERIM = Path("data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)


# Use CSV+gzip for portability (no pyarrow required)
def save_interim(df, fname):
    path = INTERIM / fname
    df.to_csv(path, index=False)
    return path


saves = {
    "nps_viis_annual.csv": viis_ann,
    "nps_buis_annual.csv": buis_ann,
    "nps_viis_monthly.csv": viis_mo,
    "nps_buis_monthly.csv": buis_mo,
    "fred_employment.csv": fred,
    "usviber_tour24.csv": tour.drop(
        columns=[c for c in tour.columns if "_dq_flag" in c], errors="ignore"
    ),
    "worldbank_arrivals.csv": wb_arr,
    "worldbank_receipts.csv": wb_rcpt,
    "census_rooms.csv": rooms,
    "usvi_cpi.csv": cpi,
}

for fname, df in saves.items():
    save_interim(df, fname)
    print(f"  Saved {fname}  ({len(df)} rows)")

# BEA tables
bea_tables = [("2.1", "bea_va_nominal"), ("2.3", "bea_va_real")]
for pattern, label in bea_tables:
    try:
        df_bea = load_bea(pattern)
        save_interim(df_bea, f"{label}.csv")
        print(f"  Saved {label}.csv  ({len(df_bea)} rows)")
    except Exception as e:
        print(f"  ⚠ Could not save {label}: {e}")

# Also save GDP share table (computed above)
save_interim(gdp_share, "bea_af_gdp_share.csv")
print(f"  Saved bea_af_gdp_share.csv  ({len(gdp_share)} rows)")

# Census combined table
census_combined = pd.concat(
    [
        naics72(c07)[
            ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
        ].assign(vintage=2007),
        naics72(c12)[
            ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
        ].assign(vintage=2012),
        naics72(c17)[
            ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
        ].assign(vintage=2017),
        naics72(c22)[
            ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
        ].assign(vintage=2022),
    ],
    ignore_index=True,
)
save_interim(census_combined, "census_naics72_combined.csv")
print(f"  Saved census_naics72_combined.csv  ({len(census_combined)} rows)")

print("\n✓ All interim datasets saved to data/interim/")

  Saved nps_viis_annual.csv  (69 rows)
  Saved nps_buis_annual.csv  (61 rows)
  Saved nps_viis_monthly.csv  (564 rows)
  Saved nps_buis_monthly.csv  (528 rows)
  Saved fred_employment.csv  (441 rows)
  Saved usviber_tour24.csv  (16 rows)
  Saved worldbank_arrivals.csv  (50 rows)
  Saved worldbank_receipts.csv  (50 rows)
  Saved census_rooms.csv  (4 rows)
  Saved usvi_cpi.csv  (13 rows)
  Saved bea_va_nominal.csv  (200 rows)
  Saved bea_va_real.csv  (200 rows)
  Saved bea_af_gdp_share.csv  (20 rows)
  Saved census_naics72_combined.csv  (16 rows)

✓ All interim datasets saved to data/interim/
